# WB-57 calibration from NASA IWG1 in-situ logs

Mirrors the G-III calibration in
`notebooks/calibration/giii/calibration.ipynb`, adapted to the
NASA 926/927 WB-57 high-altitude research data delivered as
concatenated `n92*_alltracks.csv` files (split into per-sortie
files via `hyplan.aircraft.split_iwg1_alltracks`).

The output of this notebook is a paste-ready `NASA_WB57()`
constructor block (§11).  The WB-57 is a high-altitude reconnaissance
twin-jet with typical cruise FL550-FL620, so the altitude bins
extend higher than the G-III but the methodology is the same.


In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hyplan import ureg
from hyplan.aircraft import load_iwg1, trim_ground_taxi, NASA_WB57

# Shared helpers live one directory up (notebooks/calibration/_common.py).
sys.path.insert(0, str(Path("..").resolve()))
from _common import (
    label_phases, apply_sortie_filters, per_bin, tas_per_bin,
    schedule_pts, evaluate_profile, summary_table,
)

DATA_DIR = Path("../../../data/wb57").resolve()
TAIL_GLOB = "n92[67]_*.txt"  # NASA 926 + 927

# Phase-label thresholds.  Same defaults as the ER-2 notebook —
# climb/descent gate at 300 fpm separates sustained vertical motion
# from autopilot ±100 ft cruise oscillation.
CLIMB_FPM   = 300.0
DESCENT_FPM = -300.0

# Sortie filters: real flights only.
MIN_DUR_MIN     = 60.0   # below this is taxi / engine run
MAX_DUR_MIN     = 600.0  # WB-57 sorties typically 4-6 hr
MIN_PEAK_ALT_FT = 35000  # WB-57 cruises high; below FL350 is a test/ferry


## 1. Load + trim ground taxi + phase-label every sortie


In [2]:
sorties = {}
skipped = []
for p in sorted(DATA_DIR.glob(TAIL_GLOB)):
    raw = load_iwg1(p)
    a = trim_ground_taxi(raw)
    a, reason = apply_sortie_filters(
        a, min_dur_min=MIN_DUR_MIN, max_dur_min=MAX_DUR_MIN,
        min_peak_alt_ft=MIN_PEAK_ALT_FT,
    )
    if reason is not None:
        skipped.append((p.stem, reason))
        continue
    sorties[p.stem] = label_phases(a, climb_fpm=CLIMB_FPM, descent_fpm=DESCENT_FPM)

summary_table(sorties, skipped, source_label=str(DATA_DIR.name))


source:            wb57
raw files:         151
valid sorties:     100
excluded:          51
excluded by reason:
    29  no airborne fixes
    13  too short
     7  low peak alt
     2  no valid altitude
date range:        2018-11-05 → 2026-04-30


,source,raw_files,valid_sorties,excluded,exclusion_reasons,date_range
0,wb57,151,100,51,"29 no airborne fixes, 13 too short, 7 low peak...",2018-11-05 → 2026-04-30


## 2. Per-sortie altitude profiles


In [3]:
fig, ax = plt.subplots(figsize=(13, 5))
for name, a in sorties.items():
    t_min = (a["timestamp"] - a["timestamp"].iloc[0]).dt.total_seconds() / 60.0
    ax.plot(t_min, a["altitude"] / 1000, lw=0.6, alpha=0.4, color="steelblue")
ax.set_xlabel("minutes from takeoff")
ax.set_ylabel("altitude (kft)")
ax.set_title(f"NASA 926/927 WB-57 altitude profiles — {len(sorties)} sorties")
ax.grid(alpha=0.3)
ax.set_ylim(0, 50)
plt.tight_layout()
plt.show()


/var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/ipykernel_99612/1025297844.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Ground tracks


In [4]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(13, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.95")
ax.add_feature(cfeature.OCEAN, facecolor="0.85")
ax.add_feature(cfeature.COASTLINE, lw=0.4)
ax.add_feature(cfeature.BORDERS, lw=0.4, ls=":")

for name, a in sorties.items():
    ax.plot(a["longitude"], a["latitude"], lw=0.4, alpha=0.4,
            color="steelblue", transform=ccrs.PlateCarree())

# Bound to the sortie footprint.
all_lat = pd.concat([a["latitude"] for a in sorties.values()])
all_lon = pd.concat([a["longitude"] for a in sorties.values()])
pad = 2
ax.set_extent([all_lon.min()-pad, all_lon.max()+pad,
               all_lat.min()-pad, all_lat.max()+pad])
ax.set_title(f"NASA 926/927 WB-57 ground tracks — {len(sorties)} sorties")
plt.tight_layout()
plt.show()


/var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/ipykernel_99612/1140265379.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Climb / descent: per-altitude-bin medians

Same active-climb / active-descent definitions used for ER-2 phase 2:
restrict to fixes with VS > 1500 fpm (or < -1500 fpm for descent) so
the per-bin medians represent pure aircraft performance, not cruise
plateaus or weight-management holds the planner should model
separately via `typical_climb_out` if needed.


In [5]:
ACTIVE_VS_THR_FPM = 1500.0
BIN_FT = 5000

climb_bins   = per_bin(sorties, "climb",   +1, ACTIVE_VS_THR_FPM, bin_ft=BIN_FT,
                       extra_cols=("tas_kt", "mach"))
descent_bins = per_bin(sorties, "descent", -1, ACTIVE_VS_THR_FPM, bin_ft=BIN_FT,
                       extra_cols=("tas_kt", "mach"))

print("Active CLIMB (VS >= 1500 fpm):")
print(climb_bins.to_string(index=False))
print()
print("Active DESCENT (VS <= -1500 fpm):")
print(descent_bins.to_string(index=False))


Active CLIMB (VS >= 1500 fpm):
 alt_bin_ft     n  vs_med  vs_p25  vs_p75  tas_med  mach_med
      -5000    84  1560.2  1524.7  1601.1    146.3       0.2
          0  6325  2137.3  1752.2  2557.3    167.9       0.3
       5000  6856  2888.9  2422.2  3264.1    184.8       0.3
      10000  7124  2843.2  2343.3  3274.7    200.0       0.3
      15000  7589  2788.0  2330.5  3144.5    216.0       0.3
      20000  8969  2679.8  2236.0  2980.0    233.9       0.4
      25000  9695  2513.7  2301.7  2809.8    254.0       0.4
      30000 11401  2194.3  1991.0  2423.7    275.8       0.5
      35000 13248  1789.5  1667.0  1978.3    299.9       0.5
      40000  4703  1614.3  1550.3  1704.3    321.8       0.6
      45000   171  1674.0  1608.2  1747.4    330.1       0.6

Active DESCENT (VS <= -1500 fpm):
 alt_bin_ft     n  vs_med  vs_p25  vs_p75  tas_med  mach_med
          0   283 -1638.2 -1907.5 -1572.5    182.5       0.3
       5000  3598 -1864.4 -2122.0 -1667.1    195.1       0.3
      10000  7513 -

## 5. Climb_profile breakpoints

VerticalProfile points pinned at:

* SL up to the highest populated bin: per 5-kft bin, the active-climb
  median from §4 (n>=30/bin).
* FL300 (certified ceiling, brochure): residual rate so the integrator
  terminates cleanly if a planner asks for ceiling.


In [6]:
CEILING_FT     = 65000           # WB-57 brochure ceiling
# Residual climb rate at the certified ceiling — the FL410 active-climb
# bin shows ~1100 fpm and the FL400 cruise-altitude bin sees no
# active-climb fixes (n=0), so we extrapolate to a small positive rate
# at the certified ceiling.  Not a regulatory definition (FAR Part 25
# service ceiling is 500 fpm at MTOW); just an anchor that lets the
# integration terminate if a planner asks for 45 kft.
CEILING_VS_FPM = 500.0

# Use the active-climb medians directly.  Don't enforce monotone-
# decreasing-from-SL: jets typically peak ROC near FL050-100 (limited
# below by 250-KCAS ATC procedures), so a clamp would push bins below
# their IQRs.  If the resulting profile shows wobble (small-sample
# bins), that's a separate decision and shouldn't be hidden in the
# calibration step.
fixed = []
for _, r in climb_bins.iterrows():
    if r["alt_bin_ft"] >= 0 and r["alt_bin_ft"] < CEILING_FT:
        fixed.append((int(r["alt_bin_ft"]), float(r["vs_med"])))
fixed.append((CEILING_FT, CEILING_VS_FPM))

print("climb_profile points (alt_ft, vs_fpm):")
for alt, vs in fixed:
    print(f"  ({alt:>5d}, {vs:6.0f})")


climb_profile points (alt_ft, vs_fpm):
  (    0,   2137)
  ( 5000,   2889)
  (10000,   2843)
  (15000,   2788)
  (20000,   2680)
  (25000,   2514)
  (30000,   2194)
  (35000,   1790)
  (40000,   1614)
  (45000,   1674)
  (65000,    500)


## 6. Descent_profile breakpoints

Same construction: per-bin medians of active descent VS, anchored at
top of approach (~300-500 ft AGL) and at cruise altitude.  The
planner will steepen this to fit short legs via
`descent_path_angle_max_deg=6.0` (same descent-shortening posture used by the calibrated ER-2 model).


In [7]:
# Use the active-descent medians directly.  Don't enforce monotone-
# increasing-with-altitude: descent VS typically peaks around FL150-200
# (where the aircraft descends near VMO in CAS), then declines in the
# upper levels (Mach-limited descent at constant M).  The previous
# monotone-from-SL clamp was forcing every bin above FL150 above its
# IQR and obscuring the real shape.
fixed_desc = []
for _, r in descent_bins.iterrows():
    if 0 <= r["alt_bin_ft"] < 45000:
        fixed_desc.append((int(r["alt_bin_ft"]), abs(float(r["vs_med"]))))
fixed_desc = sorted(fixed_desc)

print("descent_profile points (alt_ft, |vs|_fpm):")
for alt, vs in fixed_desc:
    print(f"  ({alt:>5d}, {vs:6.0f})")


descent_profile points (alt_ft, |vs|_fpm):
  (    0,   1638)
  ( 5000,   1864)
  (10000,   1949)
  (15000,   2004)
  (20000,   1996)
  (25000,   2196)
  (30000,   2396)
  (35000,   2463)
  (40000,   2191)


## 7. TAS schedules: climb / cruise / descent

Three independent TAS-vs-altitude schedules, derived from per-phase
medians of the same IWG1 fixes that drove the climb_profile and
descent_profile.  At the same altitude the three phases differ
materially — at FL300 cruise TAS is +39 kt over climb; at FL400 the
descent is +26 kt over climb — so a single schedule shared across
phases (or a fixed-offset derivation like
`_descent_schedule_from_cruise(cruise, 49)`) under-reads cruise TAS
during cruise and mis-models descent.


In [8]:
climb_tas   = tas_per_bin(sorties, ["climb"],   bin_ft=BIN_FT, n_min=50)
cruise_tas  = tas_per_bin(sorties, ["cruise"],  bin_ft=BIN_FT, n_min=50)
descent_tas = tas_per_bin(sorties, ["descent"], bin_ft=BIN_FT, n_min=50)

print("Climb TAS:")
print(climb_tas.to_string(index=False))
print()
print("Cruise TAS:")
print(cruise_tas.to_string(index=False))
print()
print("Descent TAS:")
print(descent_tas.to_string(index=False))


# Climb: SL rotation -> ceiling, climb-phase medians.  Anchor SL at
# typical jet rotation TAS (~150 kt) since the SL climb-phase bin is
# contaminated by takeoff-roll fixes still accelerating.
ROTATION_TAS_KT = 150
climb_pts = [(0, ROTATION_TAS_KT)] + schedule_pts(
    climb_tas, [10000, 20000, 30000, 40000, 50000, 60000], n_min=200
)

# Cruise: WB-57 cruises high — typical band FL500-FL620.  Below
# FL500 cruise-labeled bins are mostly transient level-offs.
cruise_pts = schedule_pts(cruise_tas, [50000, 55000, 60000, 62000], n_min=200)

# Descent: low-altitude anchor + descent-phase medians up to
# cruise ceiling.
descent_pts = schedule_pts(descent_tas, [10000, 20000, 30000, 40000, 50000, 60000], n_min=200)
descent_pts = [(0, 140)] + descent_pts

print()
print("Climb schedule  (alt_ft, tas_kt):", climb_pts)
print("Cruise schedule (alt_ft, tas_kt):", cruise_pts)
print("Descent schedule(alt_ft, tas_kt):", descent_pts)



Climb TAS:
 alt_bin_ft     n  tas_med  mach_med
      -5000  1818     96.6       0.2
          0 10970    167.2       0.3
       5000  7514    185.8       0.3
      10000  8733    200.4       0.3
      15000 10732    213.5       0.3
      20000 11937    233.6       0.4
      25000 11095    253.8       0.4
      30000 13073    276.3       0.5
      35000 15531    301.4       0.5
      40000 21909    330.3       0.6
      45000 15763    356.4       0.6
      50000  5870    373.5       0.7
      55000  2991    375.8       0.7
      60000   257    356.7       0.6

Cruise TAS:
 alt_bin_ft      n  tas_med  mach_med
      -5000   5602     33.8       0.0
          0  28833    145.4       0.2
       5000   7052    182.2       0.3
      10000   3813    195.3       0.3
      15000  48561    200.5       0.3
      20000  35045    198.9       0.3
      25000   8325    251.2       0.4
      30000   5859    252.8       0.4
      35000   4173    299.4       0.5
      40000 148608    337.0       0.6
   

## 8. Bank-angle analysis

`max_bank_deg` sets the minimum turn radius the Dubins planner uses;
it should reflect the *operational maximum* the aircraft is willing
to use during survey-line transitions, not the typical-mix median.

The 40k turn-fix sample (|Roll| > 5°) is dominated by small in-cruise
course corrections, so its median understates the bank used during
real maneuvers.  Use p90 instead — it captures the operational
ceiling without touching the steep-turn / emergency envelope.


In [9]:
ROLL_GATE_DEG = 5.0
banks = []
for a in sorties.values():
    abs_roll = a["roll_deg"].abs()
    in_turn = abs_roll > ROLL_GATE_DEG
    banks.append(abs_roll[in_turn])
all_banks = pd.concat(banks).dropna()
print(f"n turn fixes: {len(all_banks):,}")
print(f"|Roll| median:  {all_banks.median():.1f}°")
print(f"|Roll| p75/p90: {all_banks.quantile(0.75):.1f}° / {all_banks.quantile(0.90):.1f}°")


n turn fixes: 156,945
|Roll| median:  18.5°
|Roll| p75/p90: 28.5° / 32.5°


## 9. Operational vs aircraft-intrinsic framing

The TOC, approach-speed, and per-sortie peak-altitude statistics
that follow describe **operational** behavior across this sortie
set: wall-clock time-to-FL500 includes pre-cruise level-offs, ATC
routing, and weight-management step climbs; per-sortie peaks
reflect actual mission profiles flown rather than the airframe
service ceiling under MTOW; approach TAS is the median final-
approach speed for the mission mix.  Aircraft-intrinsic
performance (climb / descent / cruise schedules in §5–§7, bank in
§8) is what the planner consumes; the §9b numbers are reviewer-
facing context.


## 9b. TOC, approach speed, service ceiling

Empirical observations to feed the SourceRecord and the
`approach_speed` / `service_ceiling` parameters.


In [10]:
# TOC = takeoff -> first FL500 fix (WB-57 typical cruise floor).
toc = []
for a in sorties.values():
    above = a[a["altitude"] >= 50000]
    if not above.empty:
        toc.append((above["timestamp"].iloc[0] - a["timestamp"].iloc[0]).total_seconds() / 60.0)
toc_s = pd.Series(toc)

# Approach speed: median TAS in the last 500 ft AGL with VS < -200 fpm.
# Tighten the window from the G-III's ±1500 ft so we get final-approach
# speed, not the wider pattern speed which inflates the median.
approach_tas = []
for a in sorties.values():
    floor = a["altitude"].min()
    sub = a[(a["altitude"] - floor < 500) & (a["altitude"] - floor > 50)
            & (a["vertical_rate"] < -200)]
    if not sub.empty:
        approach_tas.append(sub["tas_kt"].median())
ap_s = pd.Series(approach_tas).dropna()

# Service ceiling: p99 of per-sortie peak altitudes.
peaks = pd.Series([a["altitude"].max() for a in sorties.values()])

print(f"TOC (takeoff -> FL500):  n={len(toc_s)}, median {toc_s.median():.1f} min, IQR {toc_s.quantile(.25):.1f}-{toc_s.quantile(.75):.1f}, range {toc_s.min():.1f}-{toc_s.max():.1f}")
print(f"Approach TAS:            n={len(ap_s)}, median {ap_s.median():.0f} kt, IQR {ap_s.quantile(.25):.0f}-{ap_s.quantile(.75):.0f}")
print(f"Per-sortie peak alt:     median {peaks.median():.0f} ft, p99 {peaks.quantile(0.99):.0f} ft, max {peaks.max():.0f} ft")


TOC (takeoff -> FL500):  n=26, median 31.5 min, IQR 25.4-114.6, range 18.6-450.7
Approach TAS:            n=84, median 117 kt, IQR 114-122
Per-sortie peak alt:     median 45412 ft, p99 63136 ft, max 64179 ft


## 10. Validate calibrated profiles against observed data

Overlay the proposed VerticalProfile breakpoints on the per-bin
medians + IQR shading.  No comparison line against the shipping
`NASA_WB57()` — every commit makes the two trivially identical, and
the IQR + medians alone tell the calibration story.


In [11]:
alt_grid = np.arange(0, 66000, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.fill_betweenx(climb_bins["alt_bin_ft"]/1000,
                 climb_bins["vs_p25"], climb_bins["vs_p75"],
                 alpha=0.25, color="steelblue", label="active-climb IQR")
ax.plot(climb_bins["vs_med"], climb_bins["alt_bin_ft"]/1000,
        "o", color="steelblue", label="active-climb median")
ax.plot(evaluate_profile(fixed, alt_grid), alt_grid/1000,
        color="C1", lw=2, label="proposed")
ax.set_xlabel("VS (fpm)"); ax.set_ylabel("altitude (kft)")
ax.set_title("Climb profile"); ax.legend(loc="upper right")
ax.grid(alpha=0.3); ax.set_xlim(0, 6000)

ax = axes[1]
ax.fill_betweenx(descent_bins["alt_bin_ft"]/1000,
                 (-descent_bins["vs_p75"]).abs(),
                 (-descent_bins["vs_p25"]).abs(),
                 alpha=0.25, color="firebrick", label="active-descent IQR")
ax.plot((-descent_bins["vs_med"]).abs(), descent_bins["alt_bin_ft"]/1000,
        "o", color="firebrick", label="active-descent median")
ax.plot(evaluate_profile(fixed_desc, alt_grid), alt_grid/1000,
        color="C1", lw=2, label="proposed")
ax.set_xlabel("|VS| (fpm)"); ax.set_ylabel("altitude (kft)")
ax.set_title("Descent profile"); ax.legend(loc="upper right")
ax.grid(alpha=0.3); ax.set_xlim(0, 6000)

plt.tight_layout()
plt.show()


/var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/ipykernel_99612/1639467942.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Code-paste-ready constructor block


In [12]:
print("# Paste into hyplan/aircraft/_models.py NASA_WB57.__init__")
print()
print("climb_profile=VerticalProfile(points=[")
for alt, vs in fixed:
    print(f"    ({alt:>5d} * ureg.feet, {vs:6.0f} * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)")
print("]),")
print()
print("descent_profile=VerticalProfile(points=[")
for alt, vs in fixed_desc:
    print(f"    ({alt:>5d} * ureg.feet, {vs:6.0f} * ureg.feet / ureg.minute),  # active-descent median (n>=30/bin)")
print("]),")
print()
print("climb_schedule=TasSchedule(points=[")
for alt, tas in climb_pts:
    print(f"    ({alt:>5d} * ureg.feet, {tas:3d} * ureg.knot),  # climb-phase median")
print("]),")
print()
print("cruise_schedule=TasSchedule(points=[")
for alt, tas in cruise_pts:
    print(f"    ({alt:>5d} * ureg.feet, {tas:3d} * ureg.knot),  # cruise-phase median")
print("]),")
print()
print("descent_schedule=TasSchedule(points=[")
for alt, tas in descent_pts:
    print(f"    ({alt:>5d} * ureg.feet, {tas:3d} * ureg.knot),  # descent-phase median")
print("]),")
print()
print(f"# Approach speed: median TAS in last-500-ft AGL descent across {len(ap_s)} sorties.")
print(f"approach_speed={int(round(ap_s.median()))} * ureg.knot,")
print()
print(f"# Service ceiling: p99 of per-sortie peak altitude across {len(sorties)} sorties.")
print(f"service_ceiling={int(round(peaks.quantile(0.99) / 1000) * 1000)} * ureg.feet,")
print()
print(f"# p90 |Roll| during turns ({len(all_banks):,} fixes, gate >5°) —")
print(f"# operational maximum, not the typical-mix median ({all_banks.median():.1f}°).")
print(f"turn_model=TurnModel(max_bank_deg={int(round(all_banks.quantile(0.90)))}.0),")
print()
print(f'sources=[SourceRecord(')
print(f'    source_type="iwg1",')
print(f'    reference="NASA 926+927 IWG1 calibration, n={len(sorties)} sorties",')
print(f'    confidence=0.85,')
print(f')],')


# Paste into hyplan/aircraft/_models.py NASA_WB57.__init__

climb_profile=VerticalProfile(points=[
    (    0 * ureg.feet,   2137 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    ( 5000 * ureg.feet,   2889 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (10000 * ureg.feet,   2843 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (15000 * ureg.feet,   2788 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (20000 * ureg.feet,   2680 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (25000 * ureg.feet,   2514 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (30000 * ureg.feet,   2194 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (35000 * ureg.feet,   1790 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (40000 * ureg.feet,   1614 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (45000 * ureg.feet,   1674 * ureg.feet / ureg.minut